In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../train.csv")

X = df.drop(columns=["Price_INR_Lakhs", "Property_ID"])
y = df["Price_INR_Lakhs"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42
)

categorical_features = [
    "City", "Locality_Type", "Property_Type", "Furnishing_Status"
]

numerical_features = [
    "BHK", "Bathrooms", "Super_Area_SqFt", "Carpet_Area_SqFt",
    "Floor_Number", "Total_Floors", "Age_of_Property",
    "Parking", "Lift_Available", "Gated_Community",
    "Distance_to_Metro_km", "Distance_to_City_Center_km"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_features),
    ("cat", categorical_pipeline, categorical_features)],
    sparse_threshold=0
)

### Early stopping on vs off

In notebook 26 the 100 and 200 iteration runs gave the same score because early stopping
stopped both at 56 trees. Here we compare both modes for each setting.

In [2]:
def evaluate(params):
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", HistGradientBoostingRegressor(random_state=42, **params))
    ])

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)

    return {
        **params,
        "MAE": mean_absolute_error(y_val, y_pred),
        "RMSE": mean_squared_error(y_val, y_pred) ** 0.5,
        "R2": r2_score(y_val, y_pred),
        "trees_used": model.named_steps["regressor"].n_iter_
    }

In [3]:
settings = [
    {"learning_rate": 0.10, "max_iter": 100},
    {"learning_rate": 0.10, "max_iter": 300},
    {"learning_rate": 0.05, "max_iter": 300},
    {"learning_rate": 0.05, "max_iter": 500},
]

results = []

for setting in settings:
    for early_stopping in [True, False]:
        results.append(evaluate({**setting, "early_stopping": early_stopping}))

early_stop_df = pd.DataFrame(results).sort_values("R2", ascending=False)
early_stop_df

,learning_rate,max_iter,early_stopping,MAE,RMSE,R2,trees_used
0,0.10,100,True,32.846547,68.672493,0.786996,56
2,0.10,300,True,32.846547,68.672493,0.786996,56
5,0.05,300,False,32.671548,68.708819,0.786771,300
4,0.05,300,True,32.834453,68.763963,0.786428,112
6,0.05,500,True,32.834453,68.763963,0.786428,112
1,0.10,100,False,32.690691,68.773679,0.786368,100
7,0.05,500,False,32.799066,68.960086,0.785208,500
3,0.10,300,False,33.060797,69.393202,0.782502,300


Without early stopping, more trees start to overfit (lr 0.10 with 300 trees is the worst).
With early stopping the model picks its own number of trees, so we keep it on and use
`learning_rate=0.05, max_iter=500` as the base for the next experiments.